# V4 Inference - Ensemble Comparison

Laeuft die Inferenz nur EINMAL pro Modell, kombiniert dann in 3 Varianten:
- **A:** DINOv2 Folds 1-3 (echte Test-Kameras), kein ConvNeXt, @392px
- **B:** Alle DINOv2 Folds, kein ConvNeXt, @392px
- **C:** DINOv2 Folds 1-3 + ConvNeXt schwach, @392px

Vergleicht welche Kombination den besten Kaggle-Score gibt.

## Config

In [1]:
import os

TAG = "T2"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

_candidates = [
    'multiview_pig_posture_recognition',
    './multiview_pig_posture_recognition',
    '/datasets/multi-view-pig-posture-recognition',
    '/multi-view-pig-posture-recognition',
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

TEST_CSV  = os.path.join(DATA_ROOT, "test.csv")
IMG_DIR   = os.path.join(DATA_ROOT, "test_images")

CKPT_DIR  = f"runs/v4_{TAG.lower()}"
CKPT_PATHS = sorted([
    os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR)
    if f.startswith("best_") and f.endswith(".pth")
])

# --- Inferenz-Aufloesung: zurueck auf Training-Aufloesung ---
INFER_IMG_SIZE = {
    "dinov2":   392,    # gleich wie Training (518 hat eher geschadet)
    "convnext": 384,
}

BATCH_SIZE   = 64
NUM_WORKERS  = 16
USE_TTA      = True
PAD_RATIO    = 0.1
NUM_CLASSES  = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

ADAPT_BN         = True
BN_ADAPT_BATCHES = 50

# --- Die 3 Varianten ---
# Jede Variante: (name, list-of-fold-numbers, weights-dict)
# Leere fold-list = alle Folds
VARIANTS = [
    ("A_dinov2_test_folds",  [1, 2, 3], {"dinov2": 1.0, "convnext": 0.0}),
    ("B_dinov2_all_folds",   [],        {"dinov2": 1.0, "convnext": 0.0}),
    ("C_dinov2_test_convnext_weak", [1, 2, 3], {"dinov2": 1.0, "convnext": 0.2}),
]

print(f"Gefundene Checkpoints: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    print(f"  - {os.path.basename(p)}")
print(f"\nVarianten: {len(VARIANTS)}")
for name, folds, weights in VARIANTS:
    print(f"  {name}: folds={folds or 'alle'}  weights={weights}")


DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Gefundene Checkpoints: 10
  - best_convnext_fold_1.pth
  - best_convnext_fold_2.pth
  - best_convnext_fold_3.pth
  - best_convnext_fold_4.pth
  - best_convnext_fold_5.pth
  - best_dinov2_fold_1.pth
  - best_dinov2_fold_2.pth
  - best_dinov2_fold_3.pth
  - best_dinov2_fold_4.pth
  - best_dinov2_fold_5.pth

Varianten: 3
  A_dinov2_test_folds: folds=[1, 2, 3]  weights={'dinov2': 1.0, 'convnext': 0.0}
  B_dinov2_all_folds: folds=alle  weights={'dinov2': 1.0, 'convnext': 0.0}
  C_dinov2_test_convnext_weak: folds=[1, 2, 3]  weights={'dinov2': 1.0, 'convnext': 0.2}


## Imports

In [2]:
import os, ast, re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


<jemalloc>: Unsupported system page size


Device: cuda


## Dataset + TTA + Loader Helpers

In [3]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.1):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform:
            crop = self.transform(crop)
        return crop, row["row_id"]


NORM = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

def build_tta_configs(img_size):
    S = img_size
    return [
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+64, S+64), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((int(S*0.75), int(S*0.75))),
                    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
    ]


def adapt_batch_norm(model, loader, device, n_batches=50):
    bn_layers = [m for m in model.modules()
                 if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.SyncBatchNorm))]
    if not bn_layers:
        return
    for bn in bn_layers:
        bn.running_mean.zero_()
        bn.running_var.fill_(1)
        bn.momentum = None
    model.train()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            if i >= n_batches: break
            model(imgs.to(device))
    model.eval()


def infer_prefix(ckpt_path):
    name = os.path.basename(ckpt_path)
    m = re.match(r"best_([a-zA-Z0-9]+)_fold_\d+\.pth", name)
    return m.group(1) if m else "unknown"


def fold_num(ckpt_path):
    m = re.search(r"fold_(\d+)\.pth$", ckpt_path)
    return int(m.group(1)) if m else -1


def load_vit_with_resampling(name, ckpt, num_classes, infer_size):
    model = timm.create_model(name, pretrained=False, num_classes=num_classes, img_size=infer_size)
    state_dict = dict(ckpt["model"])
    if "pos_embed" in state_dict:
        old_pe = state_dict["pos_embed"]
        if old_pe.shape != model.pos_embed.shape:
            try:
                from timm.layers import resample_abs_pos_embed
            except ImportError:
                from timm.models.layers import resample_abs_pos_embed
            num_prefix = getattr(model, "num_prefix_tokens", 1)
            state_dict["pos_embed"] = resample_abs_pos_embed(
                old_pe, new_size=model.patch_embed.grid_size, num_prefix_tokens=num_prefix
            )
    model.load_state_dict(state_dict, strict=False)
    return model


@torch.no_grad()
def predict_tta(model, df, img_dir, tta_configs, batch_size):
    all_probs = []
    for i, (tf, is_flipped) in enumerate(tta_configs):
        ds = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(tta_configs)}", leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            p = F.softmax(logits, dim=1).cpu().numpy()
            if is_flipped:
                p[:, [0, 1]] = p[:, [1, 0]]
            probs.append(p)
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)

print("Helpers geladen.")


Helpers geladen.


## Alle Modelle EINMAL inferenzen und Probs speichern

Dauert lang (10 Modelle * 6 TTAs). Danach sind die 3 Varianten nur noch Gewichts-Kombinationen (Sekunden).

In [4]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test-Instanzen: {len(test_df)}")

# Pro Modell: prefix, fold_num, probs (N, 5)
model_probs = []  # list of dicts

for idx, path in enumerate(CKPT_PATHS):
    prefix = infer_prefix(path)
    fnum = fold_num(path)
    print(f"\nModell {idx+1}/{len(CKPT_PATHS)}: {os.path.basename(path)} (arch={prefix}, fold={fnum})")

    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", "unknown")
    infer_size = INFER_IMG_SIZE.get(prefix, 392)

    print(f"  {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}  |  Infer@{infer_size}px")

    try:
        model = load_vit_with_resampling(name, ckpt, NUM_CLASSES, infer_size)
    except TypeError:
        model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
        model.load_state_dict(ckpt["model"])

    model.to(DEVICE).eval()

    if ADAPT_BN:
        bn_tf = T.Compose([
            T.Resize((infer_size, infer_size), interpolation=T.InterpolationMode.BICUBIC),
            T.ToTensor(), T.Normalize(*NORM),
        ])
        bn_ds = PigTestDataset(test_df, IMG_DIR, transform=bn_tf, pad_ratio=PAD_RATIO)
        bn_loader = DataLoader(bn_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
        adapt_batch_norm(model, bn_loader, DEVICE, n_batches=BN_ADAPT_BATCHES)

    tta_cfg = build_tta_configs(infer_size) if USE_TTA else [build_tta_configs(infer_size)[0]]
    probs = predict_tta(model, test_df, IMG_DIR, tta_cfg, BATCH_SIZE)

    model_probs.append({
        "prefix": prefix,
        "fold": fnum,
        "probs": probs,
        "val_f1": ckpt.get("val_f1", 0),
        "path": path,
    })

    del model
    torch.cuda.empty_cache()

print(f"\n{len(model_probs)} Modelle inferenziert, Probs gespeichert.")


Test-Instanzen: 11708

Modell 1/10: best_convnext_fold_1.pth (arch=convnext, fold=1)
  convnext_base.fb_in22k_ft_in1k  |  Val F1: 0.7042  |  Infer@384px


  TTA 1/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 2/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 3/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/183 [00:00<?, ?it/s]

...

  TTA 3/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 5/6:   0%|          | 0/183 [00:00<?, ?it/s]

  TTA 6/6:   0%|          | 0/183 [00:00<?, ?it/s]


10 Modelle inferenziert, Probs gespeichert.


## Varianten kombinieren + Submissions schreiben

In [5]:
results = []

for variant_name, use_folds, weights in VARIANTS:
    print(f"\n{'='*60}")
    print(f"  Variante: {variant_name}")
    print(f"{'='*60}")
    print(f"  Folds: {use_folds or 'alle'}  |  Weights: {weights}")

    selected = []
    total_weight_sum = 0.0
    for m in model_probs:
        # Fold-Filter
        if use_folds and m["fold"] not in use_folds:
            continue
        w = weights.get(m["prefix"], 0.0)
        if w <= 0:
            continue
        selected.append((m, w))
        total_weight_sum += w

    if not selected:
        print(f"  KEINE Modelle nach Filter!")
        continue

    print(f"  Aktive Modelle ({len(selected)}):")
    for m, w in selected:
        print(f"    - {os.path.basename(m['path'])} (f1={m['val_f1']:.4f})  weight={w}")

    weighted_probs = np.zeros_like(selected[0][0]["probs"])
    for m, w in selected:
        weighted_probs += m["probs"] * w
    final_probs = weighted_probs / total_weight_sum
    predictions = final_probs.argmax(axis=1)

    out_file = f"{TAG}_v4_{variant_name}.csv"
    submission = pd.DataFrame({
        "row_id": test_df["row_id"].values,
        "class_id": predictions.astype(int),
    })
    submission.to_csv(out_file, index=False)

    print(f"\n  Submission: {out_file}")
    print(f"  Verteilung:")
    for c in range(NUM_CLASSES):
        cnt = (submission["class_id"] == c).sum()
        pct = 100 * cnt / len(submission)
        bar = "#" * int(30 * cnt / len(submission))
        print(f"    {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5} ({pct:.1f}%)")

    results.append({
        "name": variant_name,
        "file": out_file,
        "n_models": len(selected),
        "distribution": {CLASS_NAMES[c]: int((submission['class_id']==c).sum()) for c in range(NUM_CLASSES)},
    })

print(f"\n\n{'='*60}")
print(f"  UEBERSICHT")
print(f"{'='*60}")
for r in results:
    print(f"  {r['name']:<35} -> {r['file']}  ({r['n_models']} Modelle)")



  Variante: A_dinov2_test_folds
  Folds: [1, 2, 3]  |  Weights: {'dinov2': 1.0, 'convnext': 0.0}
  Aktive Modelle (3):
    - best_dinov2_fold_1.pth (f1=0.8167)  weight=1.0
    - best_dinov2_fold_2.pth (f1=0.8685)  weight=1.0
    - best_dinov2_fold_3.pth (f1=0.8342)  weight=1.0

  Submission: T2_v4_A_dinov2_test_folds.csv
  Verteilung:
    0 - Lateral_lying_left     ###                             1293 (11.0%)
    1 - Lateral_lying_right    ###                             1539 (13.1%)
    2 - Sitting                #                                467 (4.0%)
    3 - Standing               ###############                 5896 (50.4%)
    4 - Sternal_lying          ######                          2513 (21.5%)

  Variante: B_dinov2_all_folds
  Folds: alle  |  Weights: {'dinov2': 1.0, 'convnext': 0.0}
  Aktive Modelle (5):
    - best_dinov2_fold_1.pth (f1=0.8167)  weight=1.0
    - best_dinov2_fold_2.pth (f1=0.8685)  weight=1.0
    - best_dinov2_fold_3.pth (f1=0.8342)  weight=1.0
    - best